In [0]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    unix_timestamp,
    abs
)

BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"
SILVER_TABLE = "workspace.default.silver_hvfhv_trips"

df_bronze = spark.table(BRONZE_TABLE)

print("Bronze table loaded")
print("Columns:", len(df_bronze.columns))

In [0]:
df_quality = (
    df_bronze

    # Timestamp quality
    .withColumn(
        "timestamp_quality_flag",
        when(
            (col("request_datetime") > col("on_scene_datetime")) |
            (col("on_scene_datetime") > col("pickup_datetime")) |
            (col("pickup_datetime") > col("dropoff_datetime")),
            lit("ISSUE")
        ).otherwise(lit("OK"))
    )

    # Financial quality
    .withColumn(
        "financial_quality_flag",
        when(
            (col("base_passenger_fare") < 0) |
            (col("driver_pay") < 0),
            lit("ISSUE")
        ).otherwise(lit("OK"))
    )
)

In [0]:
display(
    df_quality.groupBy("timestamp_quality_flag").count()
)

In [0]:
# ---------------------------------------
# SECTION 3: DERIVED OPERATIONAL METRICS
# ---------------------------------------

df_derived = (
    df_quality

    # Customer waiting time
    .withColumn(
        "customer_wait_seconds",
        when(
            col("pickup_datetime") >= col("request_datetime"),
            unix_timestamp("pickup_datetime")
            - unix_timestamp("request_datetime")
        )
    )

    # Driver response time
    .withColumn(
        "driver_response_seconds",
        when(
            col("on_scene_datetime") >= col("request_datetime"),
            unix_timestamp("on_scene_datetime")
            - unix_timestamp("request_datetime")
        )
    )

    # Trip duration calculated independently from timestamps
    .withColumn(
        "calculated_trip_time_seconds",
        when(
            col("dropoff_datetime") >= col("pickup_datetime"),
            unix_timestamp("dropoff_datetime")
            - unix_timestamp("pickup_datetime")
        )
    )

    # Compare source trip_time against calculated duration
    .withColumn(
        "trip_time_consistent",
        when(
            abs(
                col("trip_time")
                - col("calculated_trip_time_seconds")
            ) <= 3,
            True
        ).otherwise(False)
    )
)

In [0]:
display(
    df_derived.select(
        "request_datetime",
        "on_scene_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "customer_wait_seconds",
        "driver_response_seconds",
        "trip_time",
        "calculated_trip_time_seconds",
        "trip_time_consistent"
    ).limit(20)
)

In [0]:
display(
    df_derived.groupBy("trip_time_consistent").count()
)

In [0]:
df_flags = (
    df_derived

    .withColumn(
        "shared_request",
        when(col("shared_request_flag") == "Y", True)
        .when(col("shared_request_flag") == "N", False)
    )

    .withColumn(
        "shared_match",
        when(col("shared_match_flag") == "Y", True)
        .when(col("shared_match_flag") == "N", False)
    )

    .withColumn(
        "access_a_ride",
        when(col("access_a_ride_flag") == "Y", True)
        .when(col("access_a_ride_flag") == "N", False)
    )

    .withColumn(
        "wav_request",
        when(col("wav_request_flag") == "Y", True)
        .when(col("wav_request_flag") == "N", False)
    )

    .withColumn(
        "wav_match",
        when(col("wav_match_flag") == "Y", True)
        .when(col("wav_match_flag") == "N", False)
    )
)

In [0]:
display(
    df_flags.select(
        "shared_request_flag",
        "shared_request",
        "shared_match_flag",
        "shared_match",
        "wav_request_flag",
        "wav_request",
        "wav_match_flag",
        "wav_match"
    ).limit(20)
)

In [0]:
df_final = (
    df_flags

    .withColumn(
        "overall_quality_status",
        when(
            (col("timestamp_quality_flag") == "ISSUE") &
            (col("financial_quality_flag") == "ISSUE"),
            "TIMESTAMP_AND_FINANCIAL_ISSUE"
        )
        .when(
            col("timestamp_quality_flag") == "ISSUE",
            "TIMESTAMP_ISSUE"
        )
        .when(
            col("financial_quality_flag") == "ISSUE",
            "FINANCIAL_ISSUE"
        )
        .when(
            col("trip_time_consistent") == False,
            "TRIP_TIME_ISSUE"
        )
        .otherwise("GOOD")
    )
)

In [0]:
display(
    df_final
    .groupBy("overall_quality_status")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
silver_columns = [
    "hvfhs_license_num",
    "dispatching_base_num",
    "originating_base_num",

    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "dropoff_datetime",

    "PULocationID",
    "DOLocationID",

    "trip_miles",
    "trip_time",

    "base_passenger_fare",
    "tolls",
    "bcf",
    "sales_tax",
    "congestion_surcharge",
    "airport_fee",
    "tips",
    "driver_pay",
    "cbd_congestion_fee",

    "shared_request",
    "shared_match",
    "access_a_ride",
    "wav_request",
    "wav_match",

    "customer_wait_seconds",
    "driver_response_seconds",
    "calculated_trip_time_seconds",
    "trip_time_consistent",

    "timestamp_quality_flag",
    "financial_quality_flag",
    "overall_quality_status",

    "_source_month",
    "_source_file",
    "_ingested_at"
]

df_silver = df_final.select(silver_columns)

In [0]:
print("Silver columns:", len(df_silver.columns))

display(
    df_silver.limit(20)
)

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("_source_month")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table written successfully")
print("Table:", SILVER_TABLE)

In [0]:
# Verify Silver table

df_silver_table = spark.table(SILVER_TABLE)

print("Silver columns:", len(df_silver_table.columns))
print("Silver column names:")
print(df_silver_table.columns)

display(
    df_silver_table
    .groupBy("overall_quality_status")
    .count()
    .orderBy("overall_quality_status")
)